<a href="https://colab.research.google.com/github/kuds/mesozoic-labs/blob/main/notebooks/google_drive_summary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training Runs Summary

Scans the Google Drive `mesozoic-labs/logs/` directory for all past training runs
and builds a summary table showing:

- **Species** and **algorithm** for each run
- **Key settings** (learning rate, net arch, seed, reward weights)
- **Stage pass/fail** status based on curriculum gate thresholds
- **Metrics** (avg reward, episode length, forward velocity, training time)

This notebook works in Google Colab (with Drive mounted) or locally if you
point `LOGS_DIR` at your local logs directory.

## 1. Setup & Mount Google Drive

In [ ]:
import os
from pathlib import Path

IN_COLAB = (
    "COLAB_GPU" in os.environ
    or "COLAB_RELEASE_TAG" in os.environ
    or os.path.exists("/content")
)

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    LOGS_DIR = Path("/content/drive/MyDrive/mesozoic-labs/logs")
else:
    # Local fallback: adjust this path if your logs are elsewhere
    LOGS_DIR = Path.cwd().parent / "logs"

print(f"Scanning logs directory: {LOGS_DIR}")
print(f"Directory exists: {LOGS_DIR.exists()}")

In [ ]:
import json
import re
from datetime import datetime

import numpy as np

try:
    import pandas as pd
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])
    import pandas as pd

## 2. Discover Runs and Parse Results

The training notebooks save runs to:
```
logs/<species>/<algorithm>_<YYYYMMDD_HHMMSS>/
    stage1/
        stage_config.json    # settings (reward weights, hyperparams, thresholds)
        evaluations.npz      # raw eval metrics (timesteps, results, ep_lengths)
        stage_summary.txt    # human-readable summary
    stage2/
        ...
    stage3/
        ...
    training_summary.txt
```

For each stage we read `stage_config.json` for the settings and curriculum
thresholds, then check `evaluations.npz` to see whether the final evaluation
metrics met those thresholds.

In [ ]:
def parse_run_dir_name(run_dir_name):
    """Extract algorithm and timestamp from a run directory name.

    Expected format: <algorithm>_<YYYYMMDD_HHMMSS>
    e.g. 'ppo_20260225_143021'
    """
    match = re.match(r"^(.+?)_(\d{8}_\d{6})$", run_dir_name)
    if not match:
        return run_dir_name, None
    algorithm = match.group(1).upper()
    try:
        timestamp = datetime.strptime(match.group(2), "%Y%m%d_%H%M%S")
    except ValueError:
        timestamp = None
    return algorithm, timestamp


def check_stage_passed(stage_dir, stage_config):
    """Check whether a stage passed its curriculum gate.

    Reads evaluations.npz and compares the final evaluation window against
    the thresholds from stage_config.json.

    Returns (passed: bool|None, metrics: dict).
    None means we couldn't determine (missing data).
    """
    eval_path = stage_dir / "evaluations.npz"
    curriculum = stage_config.get("curriculum", {})
    min_reward = curriculum.get("min_avg_reward")
    min_length = curriculum.get("min_avg_episode_length")
    min_success_rate = curriculum.get("min_success_rate")

    metrics = {
        "avg_reward": None,
        "std_reward": None,
        "avg_episode_length": None,
        "timesteps_trained": None,
    }

    if not eval_path.exists():
        return None, metrics

    try:
        data = np.load(eval_path)
        results = data["results"]       # shape: (n_evals, n_episodes)
        timesteps = data["timesteps"]   # shape: (n_evals,)
    except Exception:
        return None, metrics

    if len(results) == 0:
        return None, metrics

    # Use the final evaluation window
    final_rewards = results[-1]
    mean_reward = float(np.mean(final_rewards))
    std_reward = float(np.std(final_rewards))
    metrics["avg_reward"] = round(mean_reward, 2)
    metrics["std_reward"] = round(std_reward, 2)
    metrics["timesteps_trained"] = int(timesteps[-1])

    # Episode lengths (may or may not be present)
    if "ep_lengths" in data:
        final_lengths = data["ep_lengths"][-1]
        mean_length = float(np.mean(final_lengths))
        metrics["avg_episode_length"] = round(mean_length, 1)
    else:
        mean_length = None

    # Determine pass/fail
    if min_reward is None and min_length is None and min_success_rate is None:
        # No thresholds defined; can't determine pass/fail
        return None, metrics

    passed = True
    if min_reward is not None and mean_reward < min_reward:
        passed = False
    if min_length is not None and mean_length is not None and mean_length < min_length:
        passed = False
    # Note: min_success_rate cannot be verified from evaluations.npz alone
    # (success data is not stored there). When set, it is logged as a
    # threshold that the curriculum system checks during training.
    if min_success_rate is not None:
        metrics["success_rate_threshold"] = min_success_rate

    return passed, metrics


def scan_run(species, run_dir):
    """Scan a single training run directory and return a list of row dicts."""
    algorithm, timestamp = parse_run_dir_name(run_dir.name)
    rows = []

    for stage_num in (1, 2, 3):
        stage_dir = run_dir / f"stage{stage_num}"
        if not stage_dir.is_dir():
            continue

        config_path = stage_dir / "stage_config.json"
        if config_path.exists():
            with open(config_path) as f:
                stage_config = json.load(f)
        else:
            stage_config = {}

        passed, metrics = check_stage_passed(stage_dir, stage_config)

        # Extract key settings from stage_config
        reward_weights = stage_config.get("reward_weights", {})
        hyperparams = stage_config.get("hyperparameters", {})
        curriculum = stage_config.get("curriculum", {})
        run_meta = stage_config.get("run", {})

        # Pull out the most useful settings for the summary
        lr = hyperparams.get("learning_rate")
        net_arch = hyperparams.get("policy_kwargs", {}).get("net_arch")
        if net_arch is None:
            net_arch = hyperparams.get("net_arch")
        batch_size = hyperparams.get("batch_size")
        gamma = hyperparams.get("gamma")

        row = {
            "species": species,
            "algorithm": algorithm,
            "run_date": timestamp.strftime("%Y-%m-%d %H:%M") if timestamp else "",
            "run_dir": run_dir.name,
            "stage": stage_num,
            "stage_name": stage_config.get("name", ""),
            "passed": passed,
            "avg_reward": metrics["avg_reward"],
            "std_reward": metrics["std_reward"],
            "avg_ep_length": metrics["avg_episode_length"],
            "timesteps": metrics["timesteps_trained"],
            "threshold_reward": curriculum.get("min_avg_reward"),
            "threshold_ep_length": curriculum.get("min_avg_episode_length"),
            "learning_rate": lr,
            "batch_size": batch_size,
            "gamma": gamma,
            "net_arch": str(net_arch) if net_arch else "",
            "seed": run_meta.get("seed", ""),
            "n_envs": run_meta.get("n_envs", ""),
            # Flatten a few key reward weights for quick comparison
            "alive_bonus": reward_weights.get("alive_bonus"),
            "energy_penalty": reward_weights.get("energy_penalty_weight"),
            "forward_vel_weight": reward_weights.get("forward_vel_weight"),
            "posture_weight": reward_weights.get("posture_weight"),
        }
        rows.append(row)

    return rows


print("Parsing functions ready.")


## 3. Scan All Runs

In [ ]:
all_rows = []

if not LOGS_DIR.exists():
    print(f"Logs directory not found: {LOGS_DIR}")
    print("Make sure Google Drive is mounted and contains mesozoic-labs/logs/.")
else:
    for species_dir in sorted(LOGS_DIR.iterdir()):
        if not species_dir.is_dir():
            continue
        species = species_dir.name
        for run_dir in sorted(species_dir.iterdir()):
            if not run_dir.is_dir():
                continue
            rows = scan_run(species, run_dir)
            all_rows.extend(rows)

df = pd.DataFrame(all_rows)
print(f"Found {len(df)} stage results across {df['run_dir'].nunique() if len(df) else 0} runs.")

## 4. Summary Table: Stage Pass/Fail by Species & Run

A pivot view showing each run as a row, with columns indicating whether
each stage passed its curriculum gate.

In [ ]:
if len(df) == 0:
    print("No runs found. Nothing to display.")
else:
    def _pass_label(val):
        if val is True:
            return "PASS"
        elif val is False:
            return "FAIL"
        return "-"

    # Build a compact pivot: one row per run, columns for each stage result
    pivot_rows = []
    for (species, run_dir), group in df.groupby(["species", "run_dir"], sort=False):
        row = {
            "species": species,
            "algorithm": group["algorithm"].iloc[0],
            "run_date": group["run_date"].iloc[0],
        }
        for _, stage_row in group.iterrows():
            sn = stage_row["stage"]
            name = stage_row["stage_name"]
            row[f"stage{sn} ({name})"] = _pass_label(stage_row["passed"])
            row[f"s{sn}_reward"] = stage_row["avg_reward"]
        pivot_rows.append(row)

    df_pivot = pd.DataFrame(pivot_rows)
    print("=== Stage Pass/Fail Summary ===")
    display(df_pivot)

## 5. Detailed Results Table

Every stage from every run, with settings and metrics side by side.

In [ ]:
if len(df) == 0:
    print("No runs found. Nothing to display.")
else:
    # Style pass/fail for readability
    def _style_passed(val):
        if val is True:
            return "background-color: #c6efce; color: #006100"
        elif val is False:
            return "background-color: #ffc7ce; color: #9c0006"
        return ""

    display_cols = [
        "species", "algorithm", "run_date", "stage", "stage_name", "passed",
        "avg_reward", "threshold_reward", "avg_ep_length", "threshold_ep_length",
        "timesteps", "learning_rate", "batch_size", "gamma", "net_arch",
        "seed", "n_envs",
        "alive_bonus", "energy_penalty", "forward_vel_weight", "posture_weight",
    ]
    # Only show columns that exist in df
    display_cols = [c for c in display_cols if c in df.columns]

    styled = (
        df[display_cols]
        .style
        .map(_style_passed, subset=["passed"])
    )
    print("=== Detailed Stage Results ===")
    display(styled)

## 6. Settings Comparison Across Runs

Compare hyperparameters and reward weights across runs for a selected
species and stage. Useful for identifying which settings led to passes.

In [ ]:
if len(df) == 0:
    print("No runs found. Nothing to display.")
else:
    # Group by species for per-species comparison
    for species, species_df in df.groupby("species"):
        print(f"\n{'='*60}")
        print(f"  {species.upper()} - Settings vs Outcome")
        print(f"{'='*60}")

        compare_cols = [
            "algorithm", "run_date", "stage", "stage_name", "passed",
            "avg_reward", "threshold_reward",
            "learning_rate", "batch_size", "gamma", "net_arch",
            "alive_bonus", "energy_penalty", "forward_vel_weight", "posture_weight",
        ]
        compare_cols = [c for c in compare_cols if c in species_df.columns]
        display(species_df[compare_cols].reset_index(drop=True))

## 7. Full Settings Deep Dive (Optional)

Expand the full `stage_config.json` for any run to inspect all reward
weights and hyperparameters.

In [ ]:
def show_full_config(species, run_dir_name, stage_num):
    """Print the full stage_config.json for a specific run and stage."""
    config_path = LOGS_DIR / species / run_dir_name / f"stage{stage_num}" / "stage_config.json"
    if not config_path.exists():
        print(f"Config not found: {config_path}")
        return
    with open(config_path) as f:
        config = json.load(f)
    print(f"\n--- {species} / {run_dir_name} / stage{stage_num} ---")
    print(json.dumps(config, indent=2))


# Example: uncomment and edit to inspect a specific run
# show_full_config("trex", "ppo_20260225_143021", 1)

## 8. Export to CSV (Optional)

Save the full results table as a CSV file for further analysis.

In [ ]:
if len(df) > 0:
    csv_path = LOGS_DIR / "runs_summary.csv"
    df.to_csv(csv_path, index=False)
    print(f"Summary exported to: {csv_path}")
else:
    print("No data to export.")